# Generate MultiSocial Chinese Dataset (Colab, NVIDIA NIM)

This notebook creates `multisocial_micro_zh.csv` from Chinese social media text using the NVIDIA NIM API (OpenAI-compatible).

## Required packages

- openai
- pandas
- tqdm
- scikit-learn

## Before running

Set environment variable in Colab runtime:

- `NVIDIA_API_KEY`

## Notes

- Uses model `qwen3-next-80b-a3b-instruct`
- Keeps strict human/machine balance by retaining only successful generation pairs
- Saves output to Google Drive path: `/content/drive/MyDrive/multisocial_outputs/multisocial_micro_zh.csv`


In [ ]:
# Install dependencies (Colab)
!pip -q install openai pandas tqdm scikit-learn

In [ ]:
import logging
import os
import time
from pathlib import Path
from typing import List, Tuple

import openai
import pandas as pd
from openai import APIConnectionError, APITimeoutError, APIStatusError, InternalServerError, RateLimitError
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# -----------------------------
# Global constants
# -----------------------------
RANDOM_SEED = 42
HUMAN_SAMPLE_SIZE = 2000
MODEL_NAME = "qwen/qwen3-next-80b-a3b-instruct"
MAX_RETRIES = 3
BASE_URL = "https://integrate.api.nvidia.com/v1"
INPUT_CSV_PATH = "/content/drive/MyDrive/multisocial_outputs/weibo_senti_100k.csv"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/multisocial_outputs/multisocial_micro_zh.csv"
SOURCE_NAME = "weibo"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("multisocial_zh")

from google.colab import userdata
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
if not NVIDIA_API_KEY:
    raise EnvironmentError(
        "Missing NVIDIA_API_KEY environment variable. "
        "Set it before running this notebook."
    )

client = openai.OpenAI(api_key=NVIDIA_API_KEY, base_url=BASE_URL)
logger.info("NVIDIA NIM client initialized.")
logger.info("Base URL: %s", BASE_URL)
logger.info("Model: %s", MODEL_NAME)

In [ ]:
# Optional tuning cell
MAX_RETRIES = 3

print(f"Model: {MODEL_NAME}")
print(f"Retries: {MAX_RETRIES}")
print(f"Input path: {INPUT_CSV_PATH}")
print(f"Output path: {OUTPUT_CSV_PATH}")

Model: qwen/qwen3-next-80b-a3b-instruct
Retries: 3
Input path: /content/drive/MyDrive/multisocial_outputs/weibo_senti_100k.csv
Output path: /content/drive/MyDrive/multisocial_outputs/multisocial_micro_zh.csv


In [ ]:
def _is_retriable_error(exc: Exception) -> bool:
    """Best-effort detection for transient/retriable API errors."""
    if isinstance(exc, (RateLimitError, APIConnectionError, APITimeoutError, InternalServerError)):
        return True

    if isinstance(exc, APIStatusError):
        status_code = getattr(exc, "status_code", None)
        if status_code == 429:
            return True
        if isinstance(status_code, int) and status_code >= 500:
            return True

    status_code = getattr(exc, "status_code", None)
    if status_code == 429:
        return True

    msg = str(exc).lower()
    return "429" in msg or "rate" in msg or "timeout" in msg or "connection" in msg


def paraphrase_chinese(text: str, client, model: str, max_retries: int = 3) -> tuple[bool, str]:
    """Paraphrase one Chinese social text with retry and exponential backoff."""
    prompt = (
        "Paraphrase the following Chinese social media post. Keep the exact same informal tone, "
        "length, hashtags, and emojis. IMPORTANT: Output the paraphrased text ONLY in Chinese. "
        "Do not translate it to English or any other language. Do not add conversational fillers. "
        "Just output the paraphrased text. Text: "
        f"{text}"
    )

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=512,
            )
            paraphrased_text = (response.choices[0].message.content or "").strip()
            if paraphrased_text:
                return True, paraphrased_text
            logger.warning("Empty response at attempt %s/%s", attempt, max_retries)
        except Exception as exc:
            retriable = _is_retriable_error(exc)
            logger.warning(
                "Paraphrasing failed at attempt %s/%s (retriable=%s): %s",
                attempt,
                max_retries,
                retriable,
                exc,
            )
            if not retriable:
                return False, ""

        if attempt < max_retries:
            backoff_seconds = 2 ** (attempt - 1)
            logger.warning("Retrying in %s second(s)...", backoff_seconds)
            time.sleep(backoff_seconds)

    return False, ""


def load_human_data(csv_path: str, n_samples: int = 2000, seed: int = 42) -> pd.DataFrame:
    """Load and sample human Chinese posts."""
    csv_file = Path(csv_path)
    if not csv_file.exists():
        raise FileNotFoundError(
            f"Input CSV not found at {csv_path}. "
            "Update INPUT_CSV_PATH to a valid file in Google Drive."
        )

    raw_df = pd.read_csv(csv_file)
    required_columns = {"label", "review"}
    missing_columns = required_columns.difference(raw_df.columns)
    if missing_columns:
        raise ValueError(
            f"Input CSV must contain columns {sorted(required_columns)}. Missing: {sorted(missing_columns)}"
        )

    text_series = raw_df["review"].astype("string").fillna("").str.strip()
    text_series = text_series[text_series != ""]

    if len(text_series) < n_samples:
        raise ValueError(
            f"Requested n_samples={n_samples}, but only {len(text_series)} non-empty rows are available."
        )

    sampled_texts = text_series.sample(n=n_samples, random_state=seed, replace=False).tolist()
    human_df = pd.DataFrame({
        "text": pd.Series(sampled_texts, dtype="string").fillna(""),
        "label": 0,
        "multi_label": "human",
        "source": SOURCE_NAME,
    })
    logger.info("Loaded %s sampled human rows.", len(human_df))
    return human_df


def generate_machine_data(human_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate machine paraphrases and keep strict successful pairs only."""
    successful_humans: List[dict] = []
    generated_rows: List[dict] = []
    success_count = 0
    dropped_count = 0

    logger.info("Starting generation for %s rows...", len(human_df))
    for text in tqdm(human_df["text"].tolist(), desc="Generating (nvidia)", unit="post"):
        original_text = str(text)
        success, paraphrased_text = paraphrase_chinese(
            text=original_text,
            client=client,
            model=MODEL_NAME,
            max_retries=MAX_RETRIES,
        )

        if success:
            successful_humans.append({
                "text": original_text,
                "label": 0,
                "multi_label": "human",
                "source": SOURCE_NAME,
            })
            generated_rows.append({
                "text": paraphrased_text,
                "label": 1,
                "multi_label": "qwen3-next-80b-a3b-instruct",
                "source": SOURCE_NAME,
            })
            success_count += 1
        else:
            dropped_count += 1

        # Be polite to API quota.
        time.sleep(1)

    logger.info(
        "Generation finished: success=%s dropped=%s total=%s",
        success_count,
        dropped_count,
        len(human_df),
    )

    if success_count == 0:
        raise RuntimeError(
            "No successful generations were produced. Check NVIDIA_API_KEY/quota and try again."
        )

    machine_df = pd.DataFrame(generated_rows)
    balanced_human_df = pd.DataFrame(successful_humans[: len(machine_df)])

    if len(machine_df) != len(balanced_human_df):
        raise RuntimeError("Human/machine mismatch after balancing.")

    return balanced_human_df, machine_df


def add_split_with_stratification(df: pd.DataFrame) -> pd.DataFrame:
    """Create stratified 80/20 train/test split by label."""
    _, test_idx = train_test_split(
        df.index,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=df["label"],
        shuffle=True,
    )

    out = df.copy()
    out["split"] = "train"
    out.loc[test_idx, "split"] = "test"
    return out


def build_final_df(human_df: pd.DataFrame, machine_df: pd.DataFrame) -> pd.DataFrame:
    if len(human_df) != len(machine_df):
        raise RuntimeError("Human and machine row counts must be equal.")

    combined_df = pd.concat([human_df, machine_df], ignore_index=True)
    combined_df["text"] = combined_df["text"].astype(str)
    combined_df["language"] = "zh"
    combined_df["length"] = combined_df["text"].apply(lambda x: len(x.split()))
    combined_df["potential_noise"] = 0
    combined_df = add_split_with_stratification(combined_df)

    final_columns = [
        "text",
        "label",
        "multi_label",
        "split",
        "language",
        "length",
        "source",
        "potential_noise",
    ]
    final_df = combined_df[final_columns]
    return final_df


def main(input_csv_path: str) -> pd.DataFrame:
    logger.info("Pipeline started (NVIDIA NIM, zh).")
    human_df = load_human_data(csv_path=input_csv_path, n_samples=HUMAN_SAMPLE_SIZE, seed=RANDOM_SEED)
    paired_human_df, machine_df = generate_machine_data(human_df)
    final_df = build_final_df(paired_human_df, machine_df)
    logger.info("Pipeline completed (NVIDIA NIM, zh).")
    return final_df

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Update if your input file differs.
input_csv_path = INPUT_CSV_PATH

Mounted at /content/drive


In [ ]:
# Run NVIDIA NIM pipeline
final_df = main(input_csv_path=input_csv_path)
display(final_df.head())

Generating (nvidia): 100%|██████████| 2000/2000 [1:29:03<00:00,  2.67s/post]


,text,label,multi_label,split,language,length,source,potential_noise
0,回复@沂蒙山区老母鸡:如何解决保鲜问题？ //@沂蒙山区老母鸡:我想办法给您快递好吗？[太开...,0,human,train,zh,5,weibo,0
1,幻想的吧，大中午的。 //@小树林s:这牛逼吹的有点大了。瑞银的老板都脸红。[哈哈],0,human,test,zh,2,weibo,0
2,#实时旅讯# 厦门鼓浪屿昨日出现海市蜃楼！[太开心] 海上升仙山，美哭了！新年刚开始就有奇观...,0,human,train,zh,5,weibo,0
3,电视里在公布通缉犯的画像，姐总感觉哪里不对啊[晕],0,human,train,zh,1,weibo,0
4,我想出去玩，好烦啊，天天都在这个楼里[怒]何日能见光明[悲伤],0,human,train,zh,1,weibo,0


In [ ]:
# Save output to required path
output_path = Path(OUTPUT_CSV_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(output_path, index=False)
logger.info("Saved dataset to %s", output_path)
logger.info(
    "Rows=%s | Human=%s | Machine=%s",
    len(final_df),
    (final_df["label"] == 0).sum(),
    (final_df["label"] == 1).sum(),
)

In [ ]:
# Quick verification checks (dynamic row count after dropping failed generations)
label_counts = final_df["label"].value_counts().to_dict()
assert set(label_counts.keys()) == {0, 1}, f"Unexpected labels found: {label_counts}"
assert label_counts[0] == label_counts[1], f"Class imbalance detected: {label_counts}"

split_counts = final_df.groupby(["split", "label"]).size().unstack(fill_value=0)
display(split_counts)
display(final_df["split"].value_counts())
print(f"Final paired rows per class: {label_counts[0]}")
print(f"Total rows: {len(final_df)}")

label,0,1
split,,
test,400,400
train,1600,1600


,count
split,
train,3200
test,800


Final paired rows per class: 2000
Total rows: 4000
